# EgoBlind-RA: Prompt-Conditioned Unified Model (Julia — Approach 2)

**Runtime**: Colab Pro A100 (40GB)  
**Persistent storage**: Google Drive  

## Drive layout
```
MyDrive/egoblind-ra-julia/
  data/
    train_labeled.csv
    test_labeled.csv
    baseline_frames/
      0/
      1/
      ...
  scripts/         
  configs/          
  output/          
  (persists across sessions)
```

## Pipeline
1. Mount Drive & install deps  
2. Convert CSVs → JSON  
3. Prepare SFT + DPO datasets  
4. SFT (~3-4 hrs)  
5. Generate DPO pairs (~1-2 hrs)  
6. DPO training (~1-2 hrs)  
7. Inference on test set  


---
## Cell 1 — Mount Drive + verify GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess

DRIVE_ROOT = '/content/drive/MyDrive/egoblind-ra-julia'

DATA_DIR    = f'{DRIVE_ROOT}/data'
LLAMA_FACTORY_DIR = f'{DRIVE_ROOT}/LLaMA-Factory'
SCRIPTS_DIR = f'{DRIVE_ROOT}/scripts'
CONFIGS_DIR = f'{DRIVE_ROOT}/configs'
OUTPUT_DIR  = f'{DRIVE_ROOT}/output'

for d in [DATA_DIR, SCRIPTS_DIR, CONFIGS_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Symlink so LLaMA-Factory can write to Drive directly
os.makedirs('{LLAMA_FACTORY_DIR}', exist_ok=True)

# Verify GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
print('Drive root exists:', os.path.isdir(DRIVE_ROOT))
print('Paths set up — ready to go.')

Mounted at /content/drive


In [3]:
import os, shutil
from tqdm import tqdm

src = f'{DATA_DIR}/baseline_frames/train'
dst = '/content/baseline_frames_train'
os.makedirs(dst, exist_ok=True)

folders = [f for f in os.listdir(src) if not f.startswith('.')]
for folder in tqdm(folders, desc='Copying frames'):
    shutil.copytree(os.path.join(src, folder), os.path.join(dst, folder), dirs_exist_ok=True)

print('Done ✓')

Copying frames: 100%|██████████| 2503/2503 [13:13<00:00,  3.15it/s]

Done ✓


---
## Cell 2 — Install dependencies

Run once per session (~8 min).

In [5]:
LLAMA_DIR = '/content/drive/MyDrive/egoblind-ra-julia/LLaMA-Factory'

!pip install -q -e $LLAMA_DIR"[torch,metrics]"
!pip install -q "transformers>=4.55.0,<=5.2.0" peft accelerate pillow bert-score

try:
    import flash_attn
    print(f'flash_attn {flash_attn.__version__} ✓')
except ImportError:
    print('flash_attn not found — standard attention will be used')

print('Done ✓')

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 119.0 MB/s eta 0:00:00
  Building editable for llamafactory (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1

---
## Cell 3 — HuggingFace login (to download Kimi-VL)

Get a token at huggingface.co/settings/tokens: read access is enough.

In [6]:
from huggingface_hub import login
login()   # will prompt for your token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


---
## Cell 4 — Convert CSVs → JSON

Only needs to run once. Outputs persist to Drive.

In [ ]:
# import pandas as pd
# import json, os

# ANSWER_COLS = ['answer0', 'answer1', 'answer2', 'answer3']

# def csv_to_records(df, split):
#     df = df[df['urgency'].isin(['urgent', 'not_urgent'])].copy()
#     print(f'  [{split}] {len(df)} valid rows after dropping ERROR rows')
#     records, urgency_labels = [], {}
#     for _, row in df.iterrows():
#         qid = str(row['question_id'])
#         answers = [
#             str(row[col]).strip()
#             for col in ANSWER_COLS
#             if pd.notna(row[col]) and str(row[col]).strip() != ''
#         ]
#         urgency = 'urgent' if str(row['urgency']).strip() == 'urgent' else 'non-urgent'
#         record = {
#             'question_id': qid,
#             'question': str(row['question']).strip(),
#             'answers': answers,
#             'video_id': str(int(row['video_name'])) if pd.notna(row['video_name']) else qid,
#             'question_type': str(row['type']) if pd.notna(row.get('type')) else '',
#             'urgency': urgency,
#         }
#         records.append(record)
#         urgency_labels[qid] = urgency
#     return records, urgency_labels

# train_df = pd.read_csv(f'{DATA_DIR}/train_labeled.csv')
# test_df  = pd.read_csv(f'{DATA_DIR}/test_labeled.csv')

# train_records, urgency_labels = csv_to_records(train_df, 'train')
# test_records,  _              = csv_to_records(test_df,  'test')

# with open(f'{DATA_DIR}/train.json', 'w') as f:
#     json.dump(train_records, f, indent=2, ensure_ascii=False)
# with open(f'{DATA_DIR}/test.json', 'w') as f:
#     json.dump(test_records, f, indent=2, ensure_ascii=False)
# with open(f'{DATA_DIR}/urgency_labels.json', 'w') as f:
#     json.dump(urgency_labels, f, indent=2)

# urgent = sum(1 for v in urgency_labels.values() if v == 'urgent')
# print(f'\nTrain: {len(train_records)} ({urgent} urgent, {len(train_records)-urgent} non-urgent)')
# print(f'Test:  {len(test_records)}')
# print('Saved to Drive ✓')

  [train] 2744 valid rows after dropping ERROR rows
  [test] 2564 valid rows after dropping ERROR rows

Train: 2744 (1474 urgent, 1270 non-urgent)
Test:  2564
Saved to Drive ✓


---
## Cell 5 — Prepare SFT + DPO datasets

Reads `train.json` + frames, writes `egoblind_combined_sft.json` and `egoblind_urgent_for_dpo.json` to Drive.  
Only needs to run once.

In [ ]:
# import json, os

# FRAMES_DIR = f'{DATA_DIR}/baseline_frames'
# MAX_FRAMES = 5

# SYSTEM_URGENT = (
#     '[URGENT] You are a real-time visual assistant for a blind user. '
#     'The user is wearing a camera and asking about their surroundings. '
#     'Respond with brief, actionable guidance. Be direct. '
#     'If you cannot determine the answer, say so clearly.'
# )
# SYSTEM_NONURGENT = (
#     '[NON-URGENT] You are a visual assistant for a blind user. '
#     'The user is wearing a camera and asking about their surroundings. '
#     'Provide a helpful, detailed answer based on the video content. '
#     'If you cannot determine the answer, say so clearly.'
# )

# def select_answer(answers, mode):
#     if not answers:
#         return "I don't know."
#     return min(answers, key=lambda a: len(a.split())) if mode == 'urgent' \
#            else max(answers, key=lambda a: len(a.split()))

# def get_frames(question_id, split='train'):
#     frame_dir = os.path.join(FRAMES_DIR, split, str(question_id))
#     if not os.path.isdir(frame_dir):
#         return None
#     frames = sorted([
#         os.path.join(frame_dir, fn)
#         for fn in os.listdir(frame_dir)
#         if fn.lower().endswith(('.jpg', '.jpeg', '.png')) and not fn.startswith('.')
#     ])
#     return frames[:MAX_FRAMES] if frames else None

# def build_entry(question, answer, system, images, question_id, all_answers, urgency):
#     if images:
#         # kimi_vl requires one <|media_pad|> token per image in the message text
#         image_tokens = ''.join(['<image>' for _ in images])
#         question_with_tokens = image_tokens + question
#     else:
#         question_with_tokens = question

#     entry = {
#         'conversations': [
#             {'from': 'human', 'value': question_with_tokens},
#             {'from': 'gpt',   'value': answer},
#         ],
#         'system': system,
#         '_meta': {
#             'question_id': question_id,
#             'all_answers': all_answers,
#             'urgency': urgency,
#         },
#     }
#     if images:
#         entry['images'] = images
#     return entry

# with open(f'{DATA_DIR}/train.json') as f:
#     data = json.load(f)

# combined_sft, with_frames, skipped_frames = [], 0, 0
# for item in data:
#     is_urgent = item['urgency'] == 'urgent'
#     answer = select_answer(item['answers'], 'urgent' if is_urgent else 'non-urgent')
#     system = SYSTEM_URGENT if is_urgent else SYSTEM_NONURGENT
#     images = get_frames(item['question_id'], split='train')
#     if images:
#         with_frames += 1
#     else:
#         skipped_frames += 1

#     entry = build_entry(
#         item['question'], answer, system, images,
#         item['question_id'], item['answers'], item['urgency']
#     )
#     combined_sft.append(entry)

# urgent_only = [e for e in combined_sft if e['_meta']['urgency'] == 'urgent']

# with open(f'{DATA_DIR}/egoblind_combined_sft.json', 'w') as f:
#     json.dump(combined_sft, f, indent=2, ensure_ascii=False)
# with open(f'{DATA_DIR}/egoblind_urgent_for_dpo.json', 'w') as f:
#     json.dump(urgent_only, f, indent=2, ensure_ascii=False)

# # Register datasets
# dataset_info = {
#     'egoblind_combined_sft': {
#         'file_name': 'egoblind_combined_sft.json',
#         'formatting': 'sharegpt',
#         'columns': {'messages': 'conversations', 'system': 'system', 'images': 'images'},
#     }
# }
# info_path = f'{DATA_DIR}/dataset_info.json'
# if os.path.exists(info_path):
#     with open(info_path) as f:
#         existing = json.load(f)
#     existing.update(dataset_info)
#     dataset_info = existing
# with open(info_path, 'w') as f:
#     json.dump(dataset_info, f, indent=2)

# print(f'Combined SFT:   {len(combined_sft)} examples')
# print(f'Urgent for DPO: {len(urgent_only)} examples')
# print(f'With frames:    {with_frames}')
# print(f'Text-only:      {skipped_frames}')
# print('Saved to Drive ✓')

KeyboardInterrupt: 

---
## Cell 6 — SFT training

Checkpoint saves every 200 steps to Drive. Note if the session dies, you can resume from the last checkpoint by setting `resume_from_checkpoint: true` in the yaml.

In [7]:
mm_plugin = f'{LLAMA_DIR}/src/llamafactory/data/mm_plugin.py'
with open(mm_plugin) as f:
    lines = f.readlines()
print(lines[34])

from transformers.video_utils import make_batched_videos



In [51]:
SFT_YAML = f"""
model_name_or_path: moonshotai/Kimi-VL-A3B-Instruct
trust_remote_code: true

stage: sft
do_train: true
finetuning_type: lora
bf16: true

lora_rank: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target: q_proj,v_proj

dataset: egoblind_combined_sft
dataset_dir: {DATA_DIR}
template: kimi_vl
cutoff_len: 2048
preprocessing_num_workers: 1

output_dir: {OUTPUT_DIR}/sft_unified
num_train_epochs: 5
per_device_train_batch_size: 1
gradient_accumulation_steps: 32
learning_rate: 2.0e-5
lr_scheduler_type: cosine
warmup_ratio: 0.05
max_grad_norm: 1.0
weight_decay: 0.1

gradient_checkpointing: true
flash_attn: auto

logging_steps: 10
save_steps: 200
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true
"""

with open(f'{CONFIGS_DIR}/sft_unified_colab.yaml', 'w') as f:
    f.write(SFT_YAML)
print('Wrote yaml ✓')

Wrote yaml ✓


In [85]:
import os
os.environ['DISABLE_VERSION_CHECK'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!DISABLE_VERSION_CHECK=1 TOKENIZERS_PARALLELISM=false PYTORCH_ALLOC_CONF=expandable_segments:True python /content/LLaMA-Factory-fresh/src/train.py {CONFIGS_DIR}/sft_unified_colab.yaml

[WARNING|2026-04-03 22:09:54] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-04-03 22:09:57] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:670] 2026-04-03 22:09:57,372 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--moonshotai--Kimi-VL-A3B-Instruct/snapshots/398eede0903cd983a2bfa0cc634e9ac1d843f375/config.json
[INFO|configuration_utils.py:670] 2026-04-03 22:09:57,512 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--moonshotai--Kimi-VL-A3B-Instruct/snapshots/398eede0903cd983a2bfa0cc634e9ac1d843f375/config.json
[INFO|configuration_utils.py:742] 2026-04-03 22:09:57,514 >> Model config KimiVLConfig {
  "architectures": [
    "KimiVL

---
## Cell 7 — Verify SFT checkpoint saved to Drive

In [86]:
import os

sft_output = f'{OUTPUT_DIR}/sft_unified'
checkpoints = [d for d in os.listdir(sft_output) if d.startswith('checkpoint')]
print('SFT checkpoints on Drive:', sorted(checkpoints))

# Find the final checkpoint (highest step number)
if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1]
    SFT_CHECKPOINT = f'{sft_output}/{latest}'
    print(f'Using checkpoint: {SFT_CHECKPOINT}')
else:
    # LLaMA-Factory may save final adapter directly
    SFT_CHECKPOINT = sft_output
    print(f'No numbered checkpoints found, using: {SFT_CHECKPOINT}')

SFT checkpoints on Drive: ['checkpoint-200', 'checkpoint-400', 'checkpoint-430']
Using checkpoint: /content/drive/MyDrive/egoblind-ra-julia/output/sft_unified/checkpoint-430


---
## Cell 8 — Generate DPO pairs (~1–2 hrs)

Runs inference on the urgent subset using the SFT adapter, scores 8 candidates per question with the composite loss, picks best/worst as chosen/rejected.

In [88]:
import os
sft_output = f'{OUTPUT_DIR}/sft_unified'
checkpoints = [d for d in os.listdir(sft_output) if d.startswith('checkpoint')]
if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1]
    SFT_CHECKPOINT = f'{sft_output}/{latest}'
else:
    SFT_CHECKPOINT = sft_output
print('SFT checkpoint:', SFT_CHECKPOINT)

SFT checkpoint: /content/drive/MyDrive/egoblind-ra-julia/output/sft_unified/checkpoint-430


In [92]:
model_file = '/root/.cache/huggingface/modules/transformers_modules/moonshotai/Kimi_hyphen_VL_hyphen_A3B_hyphen_Instruct/398eede0903cd983a2bfa0cc634e9ac1d843f375/modeling_kimi_vl.py'

with open(model_file) as f:
    src = f.read()

src = src.replace(
    'past_key_values_length = past_key_values.get_usable_length(seq_length)',
    'past_key_values_length = past_key_values.get_seq_length() if hasattr(past_key_values, "get_seq_length") else past_key_values.get_usable_length(seq_length)'
)

with open(model_file, 'w') as f:
    f.write(src)
print('Patched ✓')

Patched ✓


In [90]:
model_file = '/root/.cache/huggingface/modules/transformers_modules/moonshotai/Kimi_hyphen_VL_hyphen_A3B_hyphen_Instruct/398eede0903cd983a2bfa0cc634e9ac1d843f375/modeling_kimi_vl.py'

with open(model_file) as f:
    src = f.read()

src = src.replace(
    'past_length = past_key_values.seen_tokens',
    'past_length = past_key_values.seen_tokens if hasattr(past_key_values, "seen_tokens") else past_key_values.get_seq_length()'
)

with open(model_file, 'w') as f:
    f.write(src)
print('Patched ✓')

Patched ✓


In [ ]:
!python {SCRIPTS_DIR}/generate_dpo_pairs.py \
    --model_path moonshotai/Kimi-VL-A3B-Instruct \
    --adapter_path {SFT_CHECKPOINT} \
    --data_path {DATA_DIR}/egoblind_urgent_for_dpo.json \
    --output_path {DATA_DIR}/egoblind_unified_dpo.json \
    --num_candidates 8 \
    --alpha 0.4 --beta 0.3 --gamma 0.3 \
    --tau_min 5 --tau_max 30

Loaded 1474 urgent examples for DPO
Loading model...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:  44% 2469/5652 [00:03<00:05, 613.81it/s, Materializing param=language_model.model.layers.13.mlp.experts.2.gate_proj.weight]

---
## Cell 9 — DPO training

In [ ]:
import os, json

# Register DPO dataset (generate_dpo_pairs.py writes dataset_info.json to the data dir,
# but we need to make sure it also has the SFT entry)
info_path = f'{DATA_DIR}/dataset_info.json'
with open(info_path) as f:
    info = json.load(f)
info['egoblind_unified_dpo'] = {
    'file_name': 'egoblind_unified_dpo.json',
    'formatting': 'sharegpt',
    'ranking': True,
    'columns': {
        'messages': 'conversations',
        'chosen': 'chosen',
        'rejected': 'rejected',
        'system': 'system',
    },
}
with open(info_path, 'w') as f:
    json.dump(info, f, indent=2)
print('dataset_info.json updated ✓')

# Write DPO yaml with Drive paths
DPO_YAML = f"""### DPO config for UNIFIED Kimi-VL model (Approach 2)
### Auto-generated for Colab Pro A100

model_name_or_path: moonshotai/Kimi-VL-A3B-Instruct
adapter_name_or_path: {SFT_CHECKPOINT}
trust_remote_code: true

stage: dpo
do_train: true
finetuning_type: lora
bf16: true
dpo_beta: 0.1
dpo_loss: sigmoid

lora_rank: 64
lora_alpha: 128
lora_dropout: 0.05
lora_target: q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj

dataset: egoblind_unified_dpo
dataset_dir: {DATA_DIR}
template: kimi_vl
cutoff_len: 4096
preprocessing_num_workers: 4

output_dir: {OUTPUT_DIR}/dpo_unified
num_train_epochs: 3
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 5.0e-6
lr_scheduler_type: cosine
warmup_ratio: 0.1
max_grad_norm: 1.0
weight_decay: 0.1

gradient_checkpointing: true
flash_attn: auto

logging_steps: 10
save_steps: 100
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true
"""

dpo_yaml_path = f'{CONFIGS_DIR}/dpo_unified_colab.yaml'
with open(dpo_yaml_path, 'w') as f:
    f.write(DPO_YAML)
print(f'Wrote {dpo_yaml_path}')

dataset_info.json updated ✓
Wrote /content/drive/MyDrive/egoblind-ra-julia/configs/dpo_unified_colab.yaml


In [ ]:
!llamafactory-cli train {CONFIGS_DIR}/dpo_unified_colab.yaml

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Traceback (most recent call last):
  File "/usr/local/bin/llamafactory-cli", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/drive/.shortcut-targets-by-id/1CZGtsXz6rWQq1LPMyCj1emC_mC88aA5L/egoblind-ra-julia/LLaMA-Factory/src/llamafactory/cli.py", line 24, in main
    launcher.launch()
  File "/content/drive/.shortcut-targets-by-id/1CZGtsXz6rWQq1LPMyCj1emC_mC88aA5L/egoblind-ra-julia/LLaMA-Factory/src/llamafactory/launcher.py", line 157, in launch
    run_exp()
  File "/content/drive/.shortcut-targets-by-id/1CZGtsXz6rWQq1LPMyCj1emC_mC88aA5L/egoblind-ra-julia/LLaMA-Factory/src/llamafactory/train/tuner.py", line 139, in run_exp
    _training_function(config={"args": args, "callbacks": callbacks})
  File "/content/drive/.shortcut-targets-by-id/1CZGtsXz6rWQq1LPMyCj1emC_mC88aA5L/egoblind-ra-julia/LLaMA-Factory/src/llamafactory/train/tuner.py", line 65, in _training_function
    mo

---
## Cell 10 — Inference on test set

In [ ]:
import os

# Find DPO checkpoint
dpo_output = f'{OUTPUT_DIR}/dpo_unified'
checkpoints = [d for d in os.listdir(dpo_output) if d.startswith('checkpoint')]
if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1]
    DPO_CHECKPOINT = f'{dpo_output}/{latest}'
else:
    DPO_CHECKPOINT = dpo_output
print('DPO checkpoint:', DPO_CHECKPOINT)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/egoblind-ra-julia/output/dpo_unified'

In [ ]:
!python {SCRIPTS_DIR}/inference.py \
    --base_model moonshotai/Kimi-VL-A3B-Instruct \
    --adapter_path {DPO_CHECKPOINT} \
    --test_data {DATA_DIR}/test.json \
    --frames_dir {DATA_DIR}/baseline_frames/test \
    --output_path {OUTPUT_DIR}/unified_predictions.json \
    --best_of_k 5

---
## Cell 11 — Quick results summary

In [ ]:
import json

with open(f'{OUTPUT_DIR}/unified_predictions.json') as f:
    preds = json.load(f)

# Only score examples that have reference answers
scored = [p for p in preds if p.get('references')]

def token_f1(pred, ref):
    p_tok = set(pred.lower().split())
    r_tok = set(ref.lower().split())
    if not p_tok or not r_tok:
        return 0.0
    overlap = p_tok & r_tok
    prec = len(overlap) / len(p_tok)
    rec  = len(overlap) / len(r_tok)
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

def best_f1(pred, refs):
    return max(token_f1(pred, r) for r in refs) if refs else 0.0

overall_f1 = sum(best_f1(p['prediction'], p['references']) for p in scored) / len(scored)

# Per urgency
for urg in ['urgent', 'non-urgent']:
    subset = [p for p in scored if p.get('urgency') == urg]
    if subset:
        avg = sum(best_f1(p['prediction'], p['references']) for p in subset) / len(subset)
        avg_len = sum(len(p['prediction'].split()) for p in subset) / len(subset)
        print(f'{urg:12s}  F1={avg:.4f}  avg_len={avg_len:.1f} tokens  (n={len(subset)})')

print(f'\nOverall F1 (scored examples): {overall_f1:.4f}')
print(f'Total predictions: {len(preds)} ({len(preds)-len(scored)} test examples without references)')

---
## Cell 12 — Session crash recovery

If the Colab session dies mid-training:

1. Re-run **Cell 1** (mount Drive)
2. Re-run **Cell 2** (install deps)
3. Re-run **Cell 3** (HF login)
4. Skip to whichever cell currently on
5. For SFT/DPO, add `resume_from_checkpoint: true` to the yaml: the trainer will pick up from the last Drive checkpoint automatically

All data and checkpoints are on Drive, so nothing is lost.

In [ ]:
# RESUME HELPER — run this after cells 1-3 if session died during SFT
# Adds resume_from_checkpoint to the SFT yaml and relaunches

import os
sft_yaml_path = f'{CONFIGS_DIR}/sft_unified_colab.yaml'
with open(sft_yaml_path) as f:
    content = f.read()
if 'resume_from_checkpoint' not in content:
    content += '\nresume_from_checkpoint: true\n'
    with open(sft_yaml_path, 'w') as f:
        f.write(content)
    print('Added resume_from_checkpoint: true')
else:
    print('Already set')

# Then re-run: !llamafactory-cli train {CONFIGS_DIR}/sft_unified_colab.yaml